In [1]:
# from google.colab import userdata
# from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

# LangChain + Gemini
# import langchain, langchain_core
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
# Recommended for demos: fast & economical
model_name = "gemini-3.5-flash"

llm = ChatGoogleGenerativeAI(
    model=model_name,
    convert_system_message_to_human=True,  # smoother behavior with system prompts
    temperature=0.7,
)
print(f" Ready with model: {model_name}")

 Ready with model: gemini-3.5-flash


# Simple sequential chain (Ideas → Summary)

In [3]:
# Step 1: generate ideas
idea_prompt = PromptTemplate.from_template(
    "Generate 3 creative ideas about: {topic}. Explain each idea to a {person}.Return them as a numbered list."
)

# Step 2: summarize the ideas
summary_prompt = PromptTemplate.from_template(
    "Summarize the ideas below in 2 friendly sentences for a {person}:\n\n{ideas}"
)

In [4]:
# Chain

idea_chain = idea_prompt | llm | StrOutputParser()

In [5]:
# Compose a chain where the output of idea_chain is fed into the summary prompt
sequential_chain = (
    {
        "ideas": idea_chain,
        "topic": RunnablePassthrough(),
        "person": RunnablePassthrough(),
    }
    | summary_prompt
    | llm
    | StrOutputParser()
)

In [6]:
topic = "using AI to improve school homework feedback"
person = "baby"
Creative_ideas = idea_chain.invoke({"topic": topic, "person": person})
summary = sequential_chain.invoke({"topic": topic, "person": person})

In [7]:
print(" Topic:", topic)
print(f"{'=' * 20}")
print(" Person:", person)
print(f"{'=' * 20}")
print(" Creative Ideas : ", Creative_ideas)
print(f"{'=' * 20}")
print(" Summary:\n", summary)

 Topic: using AI to improve school homework feedback
 Person: baby
 Creative Ideas :  Here are 3 creative ideas for using AI to improve homework feedback, explained just for a baby!

1. **The Friendly Talking Toy Friend**
*The Idea:* An AI robot mascot that sits with kids while they do homework and gives instant, gentle hints instead of waiting for a grade days later.
*Explained to a baby:* Imagine a shiny teddy bear friend sitting next to you! When you draw a silly shape, the bear claps its hands and goes, "Yay! Almost! Try adding one more scribble right here!" It gives you big happy high-fives right away so you never feel confused. Yay, happy bear!

2. **Bedtime Story Feedback**
*The Idea:* AI turns a teacher’s correction notes into personalized audio bedtime stories where the student is the hero fixing their mistakes.
*Explained to a baby:* After you play with your paper, a magic computer turns your homework into a sweet story! A cozy voice whispers, "Once upon a time, baby was a br